
# Rapport d'Audit Qualité - Projet PricePulse AI

**Rôle :** Quality Engineer (QE)

**Auteur :** MOURAD Salma

**Module ciblé :** DevOps / Backend (Orchestration & Infrastructure)

**Date :** 2026

#### Introduction et objectifs:
Ce document recense les étapes de contrôle qualité effectuées sur le travail de l'équipe Backend (Tarik) et DevOps (Salim).

L'objectif est d'assurer les trois piliers du projet ("**North Star Metrics**") :

* Accuracy (Précision) : Le code est-il fiable ?
* Resilience (Résilience) : Le système gère-t-il les erreurs ?
* Trust (Confiance) : Le code est-il propre et sécurisé ?

* **Dépôt source :** GitHub (Tarik & Salim)
* **Outils d'analyse installés :** flake8 (Style), bandit (Sécurité).
* **Statut de l'installation :** Opérationnel (via pip3).

#### Audit de Propreté du Code (Linter)
* **Outil :** Flake8
* **Nombre total d'erreurs trouvées :**  92

### Analyse détaillée des erreurs (Linter)
Après exclusion des dossiers système (`.venv`), nous trouvons **92 erreurs** dans le code source.

**Top des erreurs détectées :**
* **Erreurs de style (E) :** [E501]
* **Erreurs de logique/import (F) :** [F401] [F541]

**Conclusion temporaire :** Le code nécessite un nettoyage (Refactoring) pour respecter les standards professionnels (PEP8).

#### Section 3 : Audit de Sécurité (Bandit)

Outil : Bandit

Lignes de code scannées : 567

Résultat global : 0 High, 1 Medium, 0 Low.

Détail du risque Medium : [Section 3 : Audit de Sécurité (Bandit)

Outil : Bandit

Lignes de code scannées : 567

Résultat global : 0 High, 1 Medium, 0 Low.

Détail du risque Medium : Test results:
>> Issue: [B108:hardcoded_tmp_directory] Probable insecure usage of temp file/directory.
   Severity: Medium   Confidence: Medium
   CWE: CWE-377 (https://cwe.mitre.org/data/definitions/377.html)
   More Info: https://bandit.readthedocs.io/en/1.9.4/plugins/b108_hardcoded_tmp_directory.html
   Location: ./reporting.py:10:37
9       # (répertoire éphémère, permissions variables selon l'image de base).
10      REPORT_DIR = os.getenv("REPORT_DIR", "/tmp")

###  Détail de la vulnérabilité Medium
* **Identifiant :** B108 (Hardcoded TMP directory)
* **Localisation :** `reporting.py`, ligne 10.
* **Description :** Utilisation d'un répertoire temporaire (`/tmp`) codé en dur comme valeur par défaut.
* **Risque :** Risque de sécurité lié aux permissions sur des systèmes partagés (CWE-377).
* **Recommandation QE :** Demander à l'équipe DevOps de s'assurer que dans l'image Docker, ce répertoire est bien isolé ou de changer la valeur par défaut pour un dossier spécifique à l'application avec des permissions restreintes.

#### Note d'Audit - Installation des outils de test :

**Problème :** pytest non reconnu par l'interpréteur Python 3.14 par défaut.

**Action QE :** Installation forcée via python3 -m pip install pytest.

Recommandation : Créer un fichier requirements.txt pour automatiser cette étape et éviter ces erreurs aux nouveaux arrivants.

Section 4 : État des Tests Automatisés

Résultat du scan : collected 0 items.

Constat : Absence critique de tests unitaires ou d'intégration dans le dépôt actuel.

Risque Majeur : "Blind Deployment" (Déploiement à l'aveugle). Toute modification du code par Tarik (Backend) peut casser l'orchestration sans que l'équipe ne s'en aperçoive.

Recommandation QE : Bloquer le passage en production tant qu'une base de tests minimale (Smoke Tests) n'est pas intégrée.


#### Audit de l'Environnement et Testabilité

Lors de la tentative de mise en place des tests, plusieurs obstacles majeurs ont été identifiés concernant la portabilité du projet.

##### Rapport d'incidents (Dépendances manquantes)

Le projet était initialement non-fonctionnel après installation standard. En tentant d'importer le module principal (main.py), les erreurs suivantes ont été corrigées manuellement :

**Bibliothèque python-dotenv :**

_Impact :_ Impossible de charger les variables d'environnement (clés API, configuration).

_Action QE:_ Installation manuelle via pip.

**Bibliothèque fastapi :**

_Impact :_ Le framework principal pour créer l'API était absent, empêchant le démarrage du serveur.

_Action QE :_ Installation manuelle via pip.

**Bibliothèque httpx :**

_Impact :_ Les modules de communication (pour les Scrapers et l'IA) ne pouvaient pas fonctionner.

_Action QE :_ Installation manuelle via pip.

#### Section 5 : Audit de l'Orchestrateur (main.py)

##### Architecture :
Utilisation de FastAPI avec Injection de Dépendances (conforme aux standards).

##### Observation sur la Résilience :

* Présence de classes d'exceptions personnalisées (MarketUnreachableError, AIAgentError). C'est un bon point pour la gestion d'erreurs spécifique.

* **Alerte Qualité :** Utilisation de caches (lru_cache) sur les clients API. Risque de conserver des connexions obsolètes si le service distant redémarre.

##### * Observation sur la Sécurité :

* Dépendance forte aux variables d'environnement (SCRAPER_API_KEY).

* **Recommandation :** Ajouter un test de validation qui vérifie la présence de ces clés avant de lancer le serveur.

#### Section 6 : Analyse de la Résilience et des Routes

_Gestion des Dépendances Externes :_

**Observation :** Mise en place de "Fallbacks" vers des Mocks si les clés API sont manquantes.

**Risque :** Risque de "Faux Positifs" en production (le système répond mais avec des données simulées).

**Recommandation QE :** Ajouter un indicateur is_mock: true dans la réponse API pour savoir si le prix est réel ou simulé.

_Observabilité (Santé du système) :_

**Observation :** Route /health très complète avec calcul de latence par composant.

**Action QE :** Cet endpoint sera utilisé pour le monitoring automatique et les tests de charge.

_Sécurité des endpoints sensibles :_

**Observation :** Protection de la génération de rapports par Header X-Admin-Token.

**Alerte :** La valeur par défaut du token est vide.

**Recommandation QE :** Forcer l'arrêt de l'application (crash au démarrage) si ADMIN_TOKEN n'est pas défini, pour éviter une faille de sécurité.

#### Section 7 : Validation du Déploiement Local

* Statut du Serveur : OPÉRATIONNEL.

* Commande : python3 -m uvicorn main:app --reload.

* Vérification visuelle : Accès réussi à l'interface Swagger (/docs).

* Observation QE : L'API est prête à recevoir des requêtes. Les routes /estimate, /health et /admin/report sont correctement exposées.


#### 8. Conclusion de l'Audit Qualité

L'audit du module **Pricing API** (Backend/DevOps) révèle un projet structurellement solide mais nécessitant des corrections critiques avant toute mise en production.

#####  Points Forts (Les succès)

* **Résilience :** Le système de "Fallback" (Mocks) permet une continuité de service même en cas d'absence de clés API.

* **Observabilité :** La route /health fournit des métriques précises (latence) essentielles pour le monitoring.

* **Sécurité :** Protection des routes sensibles par Token.

####  Points de Vigilance (Les risques)

* **Dépendances :** Absence de fichier requirements.txt complet, rendant le projet difficile à installer sans intervention manuelle (Audit Section 5).

* **Qualité du Code :** 92 alertes de style (Flake8) qui nuisent à la maintenabilité sur le long terme.

* **Tests :** Absence totale de tests unitaires fournis par les développeurs (corrigé temporairement par le QE avec un Smoke Test).

#####  Recommandations prioritaires

1. Créer un requirements.txt incluant python-dotenv, fastapi, httpx et uvicorn.

3. Automatiser le Linter : Intégrer flake8 dans le workflow GitHub pour forcer Tarik à nettoyer son code.

5. Vérification de Production : Interdire le statut "degraded" en production si les services réels sont obligatoires.